# Optuna 튜닝 및 최종 평가

베이스라인 비교에서 가장 안정적이었던 LightGBM 을 두 타겟 각각에 대해 Optuna 로 5-fold CV RMSE 를 최소화하도록 튜닝하고, 베이스라인 대비 향상 폭을 정리합니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
N_TRIALS = 50
N_SPLITS = 5

## 전처리 결과 로드

In [ ]:
PROCESSED_DIR = Path("data/processed")

X_yield = pd.read_parquet(PROCESSED_DIR / "X_yield.parquet")
X_energy = pd.read_parquet(PROCESSED_DIR / "X_energy.parquet")
Y = pd.read_parquet(PROCESSED_DIR / "Y.parquet")
baseline_yield = pd.read_csv(PROCESSED_DIR / "baseline_yield.csv")
baseline_energy = pd.read_csv(PROCESSED_DIR / "baseline_energy.csv")

## CV RMSE 계산 함수

In [ ]:
def cv_rmse(features: pd.DataFrame, target: pd.Series, params: dict, n_splits: int = N_SPLITS, seed: int = SEED) -> float:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rmses = []
    for train_idx, valid_idx in kf.split(features):
        X_tr, X_va = features.iloc[train_idx], features.iloc[valid_idx]
        y_tr, y_va = target.iloc[train_idx], target.iloc[valid_idx]
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[])
        rmses.append(np.sqrt(mean_squared_error(y_va, model.predict(X_va))))
    return float(np.mean(rmses))

## Objective 정의

In [ ]:
def make_objective(features: pd.DataFrame, target: pd.Series):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "max_depth": trial.suggest_int("max_depth", 4, 16),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "random_state": SEED,
            "verbose": -1,
        }
        return cv_rmse(features, target, params)

    return objective

## 두 타겟에 대해 튜닝

In [ ]:
def tune_target(features: pd.DataFrame, target: pd.Series, study_name: str) -> optuna.Study:
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler, study_name=study_name)
    study.optimize(make_objective(features, target), n_trials=N_TRIALS, show_progress_bar=False)
    return study


study_yield = tune_target(X_yield, Y["outtrn_cumsum"], "yield")
study_energy = tune_target(X_energy, Y["HeatingEnergyUsage_cumsum"], "energy")

print(f"yield  best CV RMSE: {study_yield.best_value:.4f}")
print(f"energy best CV RMSE: {study_energy.best_value:.4f}")

## 베이스라인 대비 향상 폭

In [ ]:
def summarize(name: str, baseline: pd.DataFrame, study: optuna.Study) -> dict:
    best_baseline = baseline.iloc[0]
    return {
        "target": name,
        "baseline_model": best_baseline["model"],
        "baseline_rmse": best_baseline["rmse_mean"],
        "tuned_rmse": study.best_value,
        "rmse_drop": best_baseline["rmse_mean"] - study.best_value,
        "rmse_drop_pct": (best_baseline["rmse_mean"] - study.best_value) / best_baseline["rmse_mean"] * 100,
    }


summary = pd.DataFrame([
    summarize("outtrn_cumsum", baseline_yield, study_yield),
    summarize("HeatingEnergyUsage_cumsum", baseline_energy, study_energy),
])
summary

## 최종 모델 학습 및 R² 확인

In [ ]:
def fit_and_score(features: pd.DataFrame, target: pd.Series, params: dict) -> tuple[LGBMRegressor, dict]:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    rmses, r2s = [], []
    for train_idx, valid_idx in kf.split(features):
        X_tr, X_va = features.iloc[train_idx], features.iloc[valid_idx]
        y_tr, y_va = target.iloc[train_idx], target.iloc[valid_idx]
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        r2s.append(r2_score(y_va, preds))

    final_model = LGBMRegressor(**params)
    final_model.fit(features, target)
    metrics = {
        "rmse_mean": float(np.mean(rmses)),
        "rmse_std": float(np.std(rmses)),
        "r2_mean": float(np.mean(r2s)),
        "r2_std": float(np.std(r2s)),
    }
    return final_model, metrics


yield_params = {**study_yield.best_params, "random_state": SEED, "verbose": -1}
energy_params = {**study_energy.best_params, "random_state": SEED, "verbose": -1}

model_yield, metrics_yield = fit_and_score(X_yield, Y["outtrn_cumsum"], yield_params)
model_energy, metrics_energy = fit_and_score(X_energy, Y["HeatingEnergyUsage_cumsum"], energy_params)

final_metrics = pd.DataFrame([
    {"target": "outtrn_cumsum", **metrics_yield},
    {"target": "HeatingEnergyUsage_cumsum", **metrics_energy},
])
final_metrics

## 시각화 — Optuna trial 진행

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, study, title in zip(axes, [study_yield, study_energy], ["outtrn_cumsum", "HeatingEnergyUsage_cumsum"]):
    values = [t.value for t in study.trials if t.value is not None]
    best = np.minimum.accumulate(values)
    ax.plot(values, alpha=0.4, label="trial")
    ax.plot(best, color="firebrick", label="best so far")
    ax.set_xlabel("trial")
    ax.set_ylabel("CV RMSE")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## 정리

- LightGBM 을 두 타겟 각각에 대해 50 trial Optuna(TPE) 로 튜닝하여 베이스라인 대비 RMSE 를 추가로 낮췄습니다.
- 5-fold CV 의 mean ± std 를 함께 기록해 안정성을 확인했습니다.
- 동일한 파이프라인을 다른 후보 모델로도 확장 가능하지만, 본 노트북에서는 베이스라인 비교 결과를 근거로 LightGBM 단일 후보에 집중했습니다.
